# Laboratorio 6: análisis de redes sociales en YouTube
## Inciso 9. Análisis de contenido y sentimiento

Este notebook resuelve 9.1–9.3: aplica un modelo apropiado para español a los 406 comentarios, compara el sentimiento cuando el tamaño de muestra lo permite y explica los hallazgos y sus límites. El análisis usa el texto original, incluidos emojis y signos, porque limpiarlos eliminaría señales útiles para sentimiento.

### Reproducibilidad y dependencias

Dependencias: `pandas`, `numpy`, `matplotlib`, `seaborn`, `networkx`, `torch`, `transformers`, `sentencepiece`, `emoji` y `langdetect`. Si faltan las últimas bibliotecas, pueden instalarse con:

```bash
python -m pip install "transformers>=4.45,<5" "sentencepiece>=0.2,<1" "emoji>=2.14,<3" "langdetect>=1.0.9,<2"
```

La primera ejecución descarga el modelo desde Hugging Face. Se fija la revisión `a2cc0f67ebd705c55191e25a05ba23d885fcc09b` para que cambios posteriores del repositorio del modelo no alteren los resultados.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
import pickle
import re
import unicodedata

import emoji
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
from networkx.algorithms import bipartite
from networkx.algorithms.community import louvain_communities
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import Markdown, display
from langdetect import DetectorFactory, LangDetectException, detect
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DetectorFactory.seed = SEED
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", 100)

C:\Users\marti\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Datos y unidades de análisis

La unidad es un comentario identificado por `comment_id`. Se une cada comentario con su video mediante `video_id`. Los nombres se usan como etiquetas y los ID permanecen como identificadores.

In [2]:
directorios = [Path.cwd(), Path.cwd().parent]
data_dir = next((p for p in directorios if (p / "youtube_comments.csv").exists() and (p / "youtube_videos.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError("No se encontraron youtube_comments.csv y youtube_videos.csv.")

comments = pd.read_csv(data_dir / "youtube_comments.csv", dtype={
    "comment_id": "string", "video_id": "string",
    "author_channel_id": "string", "channel_id": "string",
})
videos = pd.read_csv(data_dir / "youtube_videos.csv", dtype={
    "video_id": "string", "channel_id": "string",
})

columnas_video = ["video_id", "title", "channel_name", "channel_id", "category", "source_query"]
datos = comments.merge(videos[columnas_video], on="video_id", how="left",
                       suffixes=("_comentario", "_video"), validate="many_to_one", indicator=True)
datos["texto_original"] = datos["text"].astype("string")

assert comments["comment_id"].is_unique and comments["comment_id"].notna().all()
assert datos["_merge"].eq("both").all() and len(datos) == len(comments)
assert datos["texto_original"].notna().all()
print(f"Comentarios: {len(datos):,}")
print(f"Videos comentados: {datos['video_id'].nunique():,}")
print(f"Canales comentados: {datos['channel_id_video'].nunique():,}")
print(f"Autores únicos: {datos['author_channel_id'].nunique():,}")

Comentarios: 406
Videos comentados: 19
Canales comentados: 8
Autores únicos: 332


### Integración con las comunidades del inciso 7

Para comparar por comunidad se reproduce exactamente el procedimiento de `inciso_5_al_8.ipynb`: proyección video–video ponderada por autores compartidos y Louvain con `resolution=1.0`, `threshold=1e-7` y `seed=0`. Una comunidad representa co-participación de audiencia, no amistad, aprobación ni similitud semántica garantizada.

In [3]:
ruta_red = data_dir / "red_bipartita.pkl"
if not ruta_red.exists():
    raise FileNotFoundError("Falta red_bipartita.pkl, generado en el inciso 4.")
with ruta_red.open("rb") as archivo:
    G = pickle.load(archivo)

nodos_video = [n for n, atributos in G.nodes(data=True) if atributos.get("tipo") == "video"]
G_videos = bipartite.weighted_projected_graph(G, nodos_video)
comunidades = louvain_communities(G_videos, weight="weight", resolution=1.0, threshold=1e-7, seed=0)
comunidades = sorted(comunidades, key=len, reverse=True)
mapa_comunidades = {n.replace("video::", "", 1): i + 1 for i, comunidad in enumerate(comunidades) for n in comunidad}
datos["comunidad"] = datos["video_id"].map(mapa_comunidades).astype("Int64")

assert datos["comunidad"].notna().all()
print(f"Comunidades reproducidas: {len(comunidades)}")
display(datos.groupby("comunidad").agg(comentarios=("comment_id", "size"), videos=("video_id", "nunique"), autores=("author_channel_id", "nunique")))

Comunidades reproducidas: 12


,comentarios,videos,autores
comunidad,,,
1,225,4,187
2,84,3,62
3,34,3,29
4,2,1,2
5,3,1,3
6,25,1,18
7,4,1,4
8,1,1,1
9,1,1,1


## 9.1 Método de sentimiento para español

Se utiliza [`pysentimiento/robertuito-sentiment-analysis`](https://huggingface.co/pysentimiento/robertuito-sentiment-analysis). Su modelo base, RoBERTuito, fue preentrenado con más de 500 millones de tuits en español y luego ajustado para sentimiento con aproximadamente 5,000 tuits de TASS 2020 de distintos dialectos. Reporta tres clases: negativa (`NEG`), neutral (`NEU`) y positiva (`POS`), con Macro-F1 publicado de 0.705 ± 0.003. Es más pertinente que VADER o TextBlob estándar porque estos últimos fueron diseñados principalmente para inglés.

Antes de inferir se normalizan señales sociales siguiendo el enfoque de `pysentimiento`: las URL se reemplazan por `url`, las menciones por `@usuario`, los hashtags conservan sus palabras, los emojis se convierten a descripciones en español, las repeticiones se acortan y la risa se normaliza. No se eliminan stopwords, negaciones, puntuación ni emojis. Cada texto se limita a 128 tokens, capacidad declarada por el tokenizer.

Además de la clase más probable, se conservan las tres probabilidades, la confianza máxima y el índice continuo `score_sentimiento = P(POS) - P(NEG)`, cuyo rango es [-1, 1]. Este índice permite comparar grupos sin tratar las etiquetas como una escala equidistante.

In [4]:
URL_RE = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
USUARIO_RE = re.compile(r"(?<!\w)@[a-zA-Z0-9_]{1,30}")
HASHTAG_RE = re.compile(r"\B#(\w+)")
REPETICION_RE = re.compile(r"(.)\1{2,}", flags=re.IGNORECASE)
RISA_RE = re.compile(r"[ja][ja]+aj[ja]+", flags=re.IGNORECASE)

def separar_camel(texto):
    return re.sub(r"([a-záéíóúüñ])([A-ZÁÉÍÓÚÜÑ])", r"\1 \2", texto)

def preprocesar_social(texto):
    texto = unicodedata.normalize("NFKC", str(texto))
    texto = HASHTAG_RE.sub(lambda m: " hashtag " + separar_camel(m.group(1)), texto)
    texto = USUARIO_RE.sub("@usuario", texto)
    texto = URL_RE.sub("url", texto)
    texto = REPETICION_RE.sub(lambda m: m.group(1) * 3, texto)
    texto = RISA_RE.sub("jaja", texto)
    texto = emoji.demojize(texto, language="es", delimiters=(" emoji ", " emoji "))
    texto = texto.replace("_", " ")
    return re.sub(r"\s+", " ", texto).strip()

datos["texto_modelo"] = datos["texto_original"].map(preprocesar_social)
display(datos[["texto_original", "texto_modelo"]].head(8))

,texto_original,texto_modelo
0,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel,Ese corrupto amigo de la vieja fiscal los tengo que verbose en la carcel
1,"Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay policías que le gusta del...","Están jóvenes porque no buscan un trabajo, tuvieron suerte que no hay policías que le gusta del ..."
2,"Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es una maquila","Me dejaron con ganas de demandar la ilegalidad de las reuniones virtuales, esto no es una maquila"
3,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la sombra.,Veremos a este mafioso de Mazariegos en la cárcel y un buen tiempo en la sombra.
4,eso es para que salga de USA por su propio pie \nque se auto deporten,eso es para que salga de USA por su propio pie que se auto deporten
5,Imagine if they had to walk back home.,Imagine if they had to walk back home.
6,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::hand-purple-blue-peace:,buenísima investigacion :hand-purple-blue-peace::hand-purple-blue-peace::hand-purple-blue-peace:
7,"Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están robando.","Lleven su lonchera, sacrifiquense un poco. Y reintevren ese dinero. O están robando."


In [5]:
MODELO = "pysentimiento/robertuito-sentiment-analysis"
REVISION = "a2cc0f67ebd705c55191e25a05ba23d885fcc09b"
MAX_TOKENS = 128

tokenizer = AutoTokenizer.from_pretrained(MODELO, revision=REVISION)
model = AutoModelForSequenceClassification.from_pretrained(MODELO, revision=REVISION)
model.eval()
dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(dispositivo)

print(f"Dispositivo: {dispositivo}")
print(f"Clases del modelo: {model.config.id2label}")
print(f"Revisión cargada: {model.config._commit_hash}")

Dispositivo: cpu
Clases del modelo: {0: 'NEG', 1: 'NEU', 2: 'POS'}
Revisión cargada: a2cc0f67ebd705c55191e25a05ba23d885fcc09b


In [6]:
def inferir_sentimiento(textos, batch_size=32):
    bloques = []
    textos = list(textos)
    with torch.inference_mode():
        for inicio in range(0, len(textos), batch_size):
            batch = tokenizer(textos[inicio:inicio + batch_size], padding=True, truncation=True,
                              max_length=MAX_TOKENS, return_tensors="pt")
            batch = {k: v.to(dispositivo) for k, v in batch.items()}
            probabilidades = torch.softmax(model(**batch).logits, dim=-1).cpu().numpy()
            bloques.append(probabilidades)
    return np.vstack(bloques)

longitudes = datos["texto_modelo"].map(lambda t: len(tokenizer(t, add_special_tokens=True)["input_ids"]))
probas = inferir_sentimiento(datos["texto_modelo"])
id_por_etiqueta = {etiqueta: indice for indice, etiqueta in model.config.id2label.items()}
datos["prob_negativo"] = probas[:, id_por_etiqueta["NEG"]]
datos["prob_neutro"] = probas[:, id_por_etiqueta["NEU"]]
datos["prob_positivo"] = probas[:, id_por_etiqueta["POS"]]
etiquetas_es = {"NEG": "Negativo", "NEU": "Neutro", "POS": "Positivo"}
predicciones = [model.config.id2label[i] for i in probas.argmax(axis=1)]
datos["sentimiento"] = pd.Categorical([etiquetas_es[x] for x in predicciones],
                                             categories=["Negativo", "Neutro", "Positivo"], ordered=True)
datos["confianza"] = probas.max(axis=1)
datos["score_sentimiento"] = datos["prob_positivo"] - datos["prob_negativo"]

assert np.allclose(probas.sum(axis=1), 1, atol=1e-5)
print(f"Comentarios truncados por superar {MAX_TOKENS} tokens: {(longitudes > MAX_TOKENS).sum()}")
print(f"Longitud máxima antes de truncar: {longitudes.max()} tokens")

Token indices sequence length is longer than the specified maximum sequence length for this model (133 > 128). Running this sequence through the model will result in indexing errors


Comentarios truncados por superar 128 tokens: 11
Longitud máxima antes de truncar: 328 tokens


### Control de idioma y confianza

`langdetect` se usa solo como diagnóstico, no para excluir comentarios: es inestable en textos muy cortos y el modelo base posee cierta capacidad español–inglés. Una etiqueta distinta de español indica mayor cautela, no un idioma confirmado. La confianza es la probabilidad máxima del modelo y tampoco equivale a exactitud observada.

In [7]:
def detectar_idioma(texto):
    try:
        return detect(str(texto))
    except LangDetectException:
        return "indeterminado"

datos["idioma_estimado"] = datos["texto_original"].map(detectar_idioma)
control_calidad = pd.DataFrame({
    "métrica": ["Comentarios analizados", "Estimados como español", "Estimados como otro idioma",
                "Confianza media", "Confianza < 0.60", "Textos truncados"],
    "resultado": [len(datos), datos["idioma_estimado"].eq("es").sum(),
                  datos["idioma_estimado"].ne("es").sum(), datos["confianza"].mean(),
                  datos["confianza"].lt(0.60).sum(), (longitudes > MAX_TOKENS).sum()],
})
display(control_calidad)
display(datos["idioma_estimado"].value_counts().rename_axis("idioma").to_frame("comentarios").head(10))

,métrica,resultado
0,Comentarios analizados,406.000000
1,Estimados como español,351.000000
2,Estimados como otro idioma,55.000000
3,Confianza media,0.808912
4,Confianza < 0.60,67.000000
5,Textos truncados,11.000000


,comentarios
idioma,
es,351
pt,15
en,11
it,6
ca,5
indeterminado,4
de,4
ro,3
so,1


### Resultado global

In [8]:
orden = ["Negativo", "Neutro", "Positivo"]
colores = {"Negativo": "#B33A3A", "Neutro": "#6B7280", "Positivo": "#25855A"}
conteo_global = datos["sentimiento"].value_counts(sort=False).reindex(orden, fill_value=0)
resumen_global = pd.DataFrame({
    "comentarios": conteo_global,
    "porcentaje": (conteo_global / len(datos) * 100).round(2),
})
display(resumen_global)
print(f"Score medio: {datos['score_sentimiento'].mean():.3f}")
print(f"Score mediano: {datos['score_sentimiento'].median():.3f}")
print(f"Confianza media: {datos['confianza'].mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(resumen_global.index, resumen_global["porcentaje"], color=[colores[x] for x in orden])
ax.set(title="Sentimiento de los comentarios observados", xlabel="Clase predicha", ylabel="Comentarios (%)")
ax.bar_label(ax.containers[0], fmt="%.1f%%")
sns.despine()
plt.tight_layout()
plt.show()

,comentarios,porcentaje
sentimiento,,
Negativo,250,61.58
Neutro,77,18.97
Positivo,79,19.46


Score medio: -0.376
Score mediano: -0.727
Confianza media: 0.809


C:\Users\marti\AppData\Local\Temp\ipykernel_9200\3717287029.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9.2 Comparaciones por video, canal, categoría y comunidad

Para evitar porcentajes excesivamente inestables, las tablas comparativas principales muestran grupos con al menos 5 comentarios (`N_MIN=5`). No es una prueba de significancia ni corrige el sesgo de selección. Se reportan `n`, autores únicos, probabilidades medias, proporciones por clase y score medio.

In [9]:
N_MIN = 5

def resumir_por_grupo(df, grupo, n_min=N_MIN):
    base = df.groupby(grupo, observed=True).agg(
        n=("comment_id", "size"), autores=("author_channel_id", "nunique"),
        prob_negativo=("prob_negativo", "mean"), prob_neutro=("prob_neutro", "mean"),
        prob_positivo=("prob_positivo", "mean"), score_medio=("score_sentimiento", "mean"),
        confianza_media=("confianza", "mean"),
    )
    proporciones = pd.crosstab(df[grupo], df["sentimiento"], normalize="index").reindex(columns=orden, fill_value=0)
    proporciones.columns = [f"pct_{x.lower()}" for x in proporciones.columns]
    return base.join(proporciones.mul(100)).query("n >= @n_min").sort_values(["n", "score_medio"], ascending=[False, True])

def grafico_apilado(resumen, etiquetas, titulo, max_grupos=12):
    columnas = ["pct_negativo", "pct_neutro", "pct_positivo"]
    plot_data = resumen.head(max_grupos).copy()
    plot_data.index = etiquetas.loc[plot_data.index]
    ax = plot_data[columnas].plot(kind="barh", stacked=True, figsize=(10, max(4, len(plot_data) * 0.45)),
                                     color=[colores[x] for x in orden])
    ax.set(title=titulo, xlabel="Comentarios (%)", ylabel="")
    ax.legend(orden, title="Sentimiento", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.invert_yaxis()
    sns.despine()
    plt.tight_layout()
    plt.show()

In [10]:
resumen_video = resumir_por_grupo(datos, "video_id")
meta_video = videos.set_index("video_id")[["title", "channel_name", "category"]]
resumen_video = resumen_video.join(meta_video)
columnas_mostrar = ["title", "channel_name", "n", "autores", "pct_negativo", "pct_neutro",
                    "pct_positivo", "score_medio", "confianza_media"]
display(resumen_video[columnas_mostrar].round(3))
etiquetas_video = meta_video["title"].str.slice(0, 58)
grafico_apilado(resumen_video, etiquetas_video, f"Sentimiento por video (n ≥ {N_MIN})")

,title,channel_name,n,autores,pct_negativo,pct_neutro,pct_positivo,score_medio,confianza_media
video_id,,,,,,,,,
n8iP75gIpmw,Qué rico come tu diputado,Quorum,161,128,80.745,13.665,5.590,-0.665,0.838
j43HgwYFKfk,La cooptación de Walter Mazariegos en la USAC,Quorum,50,49,58.000,20.000,22.000,-0.339,0.803
6W4u8sGEnGM,Inician los trabajos de recuperación del Puente Belice II.,Gobierno de la República de Guatemala,45,32,40.000,31.111,28.889,-0.105,0.764
OkXlHx0hx-8,EE.UU. envía a mexicanos deportados a Guatemala antes de su regreso a México | Noticias Telemundo,Noticias Telemundo,25,18,72.000,20.000,8.000,-0.487,0.724
PjmxCj-a9Hg,Conferencia de Prensa del Gobierno de Guatemala. #LaRondaGt,Gobierno de la República de Guatemala,25,19,64.000,20.000,16.000,-0.422,0.787
lj983NWyAQY,Plan 2032 Ciudad de Guatemala,Municipalidad de Guatemala,25,25,8.000,12.000,80.000,0.605,0.835
yLZS3JiEBg8,Arroz con pollo a la MONOPOLIO,Quorum,16,16,56.250,12.500,31.250,-0.234,0.887
06mFNPU0aB8,Capturan a presuntos delincuentes disfrazados de mujer señalados de cometer asalto,Noti7,14,13,28.571,50.000,21.429,-0.172,0.733
ndAZjHqzzT8,Internet: escoger el menos malo,Quorum,12,10,33.333,33.333,33.333,-0.005,0.760


C:\Users\marti\AppData\Local\Temp\ipykernel_9200\2702691149.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
resumen_canal = resumir_por_grupo(datos, "channel_id_video")
meta_canal = videos.drop_duplicates("channel_id").set_index("channel_id")["channel_name"]
resumen_canal = resumen_canal.join(meta_canal.rename("canal"))
print(f"Canales con al menos {N_MIN} comentarios:")
display(resumen_canal[["canal", "n", "autores", "pct_negativo", "pct_neutro", "pct_positivo", "score_medio"]].round(3))
grafico_apilado(resumen_canal, meta_canal, f"Sentimiento por canal (n ≥ {N_MIN})")

resumen_categoria = resumir_por_grupo(datos, "category")
print(f"Categorías de YouTube con al menos {N_MIN} comentarios:")
display(resumen_categoria.round(3))

Canales con al menos 5 comentarios:


,canal,n,autores,pct_negativo,pct_neutro,pct_positivo,score_medio
channel_id_video,,,,,,,
UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,256,214,69.922,15.625,14.453,-0.497
UC-ZtFBHzgkYFh7whZkuR_Ag,Gobierno de la República de Guatemala,70,50,48.571,27.143,24.286,-0.218
UCRwA1NUcUnwsly35ikGhp0A,Noticias Telemundo,25,18,72.000,20.000,8.000,-0.487
UCXa6LVjBzUtmcM3cwbd9e2w,Municipalidad de Guatemala,25,25,8.000,12.000,80.000,0.605
UCVpSRoZgngfSL03Nlbjtq9A,Noti7,14,13,28.571,50.000,21.429,-0.172
UCEepOY2svuxbsJ-zsAPT9Fg,PrensaLibreOficial,7,7,85.714,14.286,0.000,-0.729
UCt5Dw7ePnn0AqPqRF8Xpjkw,TN23 Guatemala,7,7,85.714,14.286,0.000,-0.631


Categorías de YouTube con al menos 5 comentarios:


C:\Users\marti\AppData\Local\Temp\ipykernel_9200\2702691149.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,n,autores,prob_negativo,prob_neutro,prob_positivo,score_medio,confianza_media,pct_negativo,pct_neutro,pct_positivo
category,,,,,,,,,,
News & Politics,335,283,0.599,0.212,0.189,-0.411,0.817,64.478,17.015,18.507
Entertainment,70,50,0.470,0.279,0.252,-0.218,0.772,48.571,27.143,24.286


In [12]:
resumen_comunidad = resumir_por_grupo(datos, "comunidad")
videos_por_comunidad = datos.groupby("comunidad")["video_id"].nunique().rename("videos")
resumen_comunidad = resumen_comunidad.join(videos_por_comunidad)
display(resumen_comunidad[["n", "videos", "autores", "pct_negativo", "pct_neutro",
                           "pct_positivo", "score_medio", "confianza_media"]].round(3))
etiquetas_comunidad = pd.Series({i: f"Comunidad {i}" for i in resumen_comunidad.index})
grafico_apilado(resumen_comunidad, etiquetas_comunidad, f"Sentimiento por comunidad Louvain (n ≥ {N_MIN})")

,n,videos,autores,pct_negativo,pct_neutro,pct_positivo,score_medio,confianza_media
comunidad,,,,,,,,
1,225,4,187,76.000,15.111,8.889,-0.593,0.827
2,84,3,62,45.238,30.952,23.810,-0.210,0.766
3,34,3,29,50.000,20.588,29.412,-0.196,0.837
6,25,1,18,72.000,20.000,8.000,-0.487,0.724
10,25,1,25,8.000,12.000,80.000,0.605,0.835


C:\Users\marti\AppData\Local\Temp\ipykernel_9200\2702691149.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Incertidumbre descriptiva

Para los grupos con al menos 10 comentarios se calcula un intervalo bootstrap percentil del 95 % para el score medio, con 2,000 remuestreos y semilla fija. Estos intervalos cuantifican variabilidad entre los comentarios observados; no convierten la muestra por consultas en una muestra probabilística de YouTube.

In [13]:
def bootstrap_score(df, grupo, n_min=10, repeticiones=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    filas = []
    for nombre, bloque in df.groupby(grupo, observed=True):
        valores = bloque["score_sentimiento"].to_numpy()
        if len(valores) < n_min:
            continue
        medias = rng.choice(valores, size=(repeticiones, len(valores)), replace=True).mean(axis=1)
        filas.append((nombre, len(valores), valores.mean(), *np.quantile(medias, [0.025, 0.975])))
    return pd.DataFrame(filas, columns=[grupo, "n", "score_medio", "IC95_inf", "IC95_sup"]).set_index(grupo)

print("Intervalos por canal:")
ic_canal = bootstrap_score(datos, "channel_id_video").join(meta_canal.rename("canal"))
display(ic_canal[["canal", "n", "score_medio", "IC95_inf", "IC95_sup"]].sort_values("score_medio").round(3))
print("Intervalos por comunidad:")
display(bootstrap_score(datos, "comunidad").sort_values("score_medio").round(3))

Intervalos por canal:


,canal,n,score_medio,IC95_inf,IC95_sup
channel_id_video,,,,,
UCE4rsXcgDb6e1-a9iTbWzfg,Quorum,256,-0.497,-0.574,-0.420
UCRwA1NUcUnwsly35ikGhp0A,Noticias Telemundo,25,-0.487,-0.661,-0.277
UC-ZtFBHzgkYFh7whZkuR_Ag,Gobierno de la República de Guatemala,70,-0.218,-0.378,-0.056
UCVpSRoZgngfSL03Nlbjtq9A,Noti7,14,-0.172,-0.459,0.143
UCXa6LVjBzUtmcM3cwbd9e2w,Municipalidad de Guatemala,25,0.605,0.373,0.791


Intervalos por comunidad:


,n,score_medio,IC95_inf,IC95_sup
comunidad,,,,
1,225,-0.593,-0.661,-0.522
6,25,-0.487,-0.663,-0.278
2,84,-0.210,-0.346,-0.068
3,34,-0.196,-0.454,0.057
10,25,0.605,0.373,0.788


### Auditoría cualitativa

Se muestran casos de alta confianza y los más ambiguos. Esta inspección no constituye validación con etiquetas humanas, pero ayuda a detectar resultados evidentemente incoherentes y recuerda que una probabilidad alta puede estar equivocada.

In [14]:
columnas_auditoria = ["sentimiento", "confianza", "score_sentimiento", "idioma_estimado", "texto_original"]
muestra_alta = (datos.sort_values("confianza", ascending=False)
                .groupby("sentimiento", observed=True, group_keys=False).head(3))
print("Ejemplos de mayor confianza por clase:")
display(muestra_alta[columnas_auditoria])
print("Ejemplos más ambiguos:")
display(datos.nsmallest(8, "confianza")[columnas_auditoria])

Ejemplos de mayor confianza por clase:


,sentimiento,confianza,score_sentimiento,idioma_estimado,texto_original
282,Negativo,0.985907,-0.983729,es,Que indignante saber como se artan estos coches y finalmente el pueblo esta ciendo dañado
99,Negativo,0.985687,-0.982183,es,"Ni el mismo lo cree, estos son sinverguenzas y se pasan de pija, cómo se sentirá la familia tene..."
310,Negativo,0.982670,-0.979824,es,"Muy cierto, mi mamá en este momento esta incapacitada de una pierna y hemos vivido lo que es cam..."
398,Positivo,0.978018,0.975605,es,"Es un proyecto extraordinario , vamos adelante mi guate hermosa!"
193,Positivo,0.976056,0.973266,es,Que bueno que Guatemala sea una cuidad bonita y ojalá todos los proyectos sean realidad. Se los ...
255,Positivo,0.974342,0.971551,es,"Gracias por este video, que importante conocer este tipo de espacios."
400,Neutro,0.885243,0.003904,es,Pregúntenle si se acuerda de la marca del vino que se toma todos los días
322,Neutro,0.885053,0.022156,es,"Pues deberían establecer viáticos cuando los trabajos sean institucionales, eso seria más legal...."
314,Neutro,0.870969,-0.022671,es,La próxima habla de la harina de trigo.


Ejemplos más ambiguos:


,sentimiento,confianza,score_sentimiento,idioma_estimado,texto_original
404,Neutro,0.354809,-0.012801,es,Este hombre esta loco
386,Negativo,0.435341,-0.290803,es,Ay ricos shucos en la calle jajaja
350,Neutro,0.441934,-0.294748,es,""" Ayudanos a luchar contra el racismo y los malos tratos; vete de regreso a tu pais""\n ..."
377,Neutro,0.470231,-0.331426,es,Mantenidos
25,Neutro,0.472742,-0.403915,es,Pero hay que ver el lado bueno si los deportan por la Frontera los carteles los secuestran para ...
340,Positivo,0.473072,0.217224,es,Despues de tanta corrpcion y desfalco por los anteriores gobiernos ladrones que esperaba Rolando...
83,Neutro,0.477862,0.115308,pt,excelente
246,Neutro,0.477862,0.115308,es,EXCELENTE


## 9.3 Interpretación de hallazgos

In [15]:
clase_dominante = resumen_global["comentarios"].idxmax()
pct_dominante = resumen_global.loc[clase_dominante, "porcentaje"]
video_mas_neg = resumen_video.sort_values("score_medio").iloc[0]
video_mas_pos = resumen_video.sort_values("score_medio", ascending=False).iloc[0]
canal_mas_neg = resumen_canal.sort_values("score_medio").iloc[0]
canal_mas_pos = resumen_canal.sort_values("score_medio", ascending=False).iloc[0]
categoria_mas_neg = resumen_categoria.sort_values("score_medio").iloc[0]
categoria_mas_pos = resumen_categoria.sort_values("score_medio", ascending=False).iloc[0]
com_mas_neg = resumen_comunidad.sort_values("score_medio").iloc[0]
com_mas_pos = resumen_comunidad.sort_values("score_medio", ascending=False).iloc[0]
pct_baja_confianza = datos["confianza"].lt(0.60).mean() * 100
pct_no_es = datos["idioma_estimado"].ne("es").mean() * 100

display(Markdown(f"""
- **Panorama general:** la clase modal es **{clase_dominante.lower()}** ({pct_dominante:.1f} %). El score medio es {datos['score_sentimiento'].mean():.3f}; por tanto, el balance agregado se interpreta como {'más favorable' if datos['score_sentimiento'].mean() > 0 else 'más desfavorable'} dentro de los comentarios recolectados, no como opinión pública general.
- **Videos (n ≥ {N_MIN}):** el balance más negativo corresponde a *{video_mas_neg['title']}* (n={int(video_mas_neg['n'])}, score={video_mas_neg['score_medio']:.3f}); el más positivo corresponde a *{video_mas_pos['title']}* (n={int(video_mas_pos['n'])}, score={video_mas_pos['score_medio']:.3f}). Esto describe reacciones al contenido y contexto de esos videos, no su calidad ni veracidad.
- **Canales (n ≥ {N_MIN}):** **{canal_mas_neg['canal']}** presenta el balance más negativo (n={int(canal_mas_neg['n'])}, score={canal_mas_neg['score_medio']:.3f}), mientras **{canal_mas_pos['canal']}** presenta el más positivo (n={int(canal_mas_pos['n'])}, score={canal_mas_pos['score_medio']:.3f}). Los canales agrupan videos de temas y fechas diferentes, por lo que la asociación no debe atribuirse causalmente al canal.
- **Categorías (n ≥ {N_MIN}):** **{categoria_mas_neg.name}** presenta el balance más negativo (n={int(categoria_mas_neg['n'])}, score={categoria_mas_neg['score_medio']:.3f}) y **{categoria_mas_pos.name}** el más positivo (n={int(categoria_mas_pos['n'])}, score={categoria_mas_pos['score_medio']:.3f}). Solo dos categorías alcanzan el umbral y sus tamaños son desiguales, por lo que `category` funciona como una comparación amplia, no como tópico específico.
- **Comunidades:** la comunidad {int(com_mas_neg.name)} tiene el score más negativo entre las que alcanzan el umbral (n={int(com_mas_neg['n'])}, score={com_mas_neg['score_medio']:.3f}); la comunidad {int(com_mas_pos.name)} tiene el más positivo (n={int(com_mas_pos['n'])}, score={com_mas_pos['score_medio']:.3f}). Sus intervalos bootstrap no incluyen cero. Estas comunidades provienen de autores compartidos y no de sentimiento; la diferencia es una caracterización posterior.
- **Incertidumbre:** {pct_baja_confianza:.1f} % de las predicciones tiene confianza menor a 0.60, {pct_no_es:.1f} % fue estimado como idioma distinto de español y {(longitudes > MAX_TOKENS).sum()} textos fueron truncados. En los intervalos por canal, Noti7 incluye cero, mientras Quorum, Noticias Telemundo y Gobierno de Guatemala quedan del lado negativo y Municipalidad de Guatemala del positivo; esto solo resume variabilidad interna de la muestra observada.
"""))


- **Panorama general:** la clase modal es **negativo** (61.6 %). El score medio es -0.376; por tanto, el balance agregado se interpreta como más desfavorable dentro de los comentarios recolectados, no como opinión pública general.
- **Videos (n ≥ 5):** el balance más negativo corresponde a *Bloqueos en Guatemala este 31 de agosto por alza en combustibles afectan rutas principales* (n=7, score=-0.729); el más positivo corresponde a *Plan 2032 Ciudad de Guatemala* (n=25, score=0.605). Esto describe reacciones al contenido y contexto de esos videos, no su calidad ni veracidad.
- **Canales (n ≥ 5):** **PrensaLibreOficial** presenta el balance más negativo (n=7, score=-0.729), mientras **Municipalidad de Guatemala** presenta el más positivo (n=25, score=0.605). Los canales agrupan videos de temas y fechas diferentes, por lo que la asociación no debe atribuirse causalmente al canal.
- **Categorías (n ≥ 5):** **News & Politics** presenta el balance más negativo (n=335, score=-0.411) y **Entertainment** el más positivo (n=70, score=-0.218). Solo dos categorías alcanzan el umbral y sus tamaños son desiguales, por lo que `category` funciona como una comparación amplia, no como tópico específico.
- **Comunidades:** la comunidad 1 tiene el score más negativo entre las que alcanzan el umbral (n=225, score=-0.593); la comunidad 10 tiene el más positivo (n=25, score=0.605). Sus intervalos bootstrap no incluyen cero. Estas comunidades provienen de autores compartidos y no de sentimiento; la diferencia es una caracterización posterior.
- **Incertidumbre:** 16.5 % de las predicciones tiene confianza menor a 0.60, 13.5 % fue estimado como idioma distinto de español y 11 textos fueron truncados. En los intervalos por canal, Noti7 incluye cero, mientras Quorum, Noticias Telemundo y Gobierno de Guatemala quedan del lado negativo y Municipalidad de Guatemala del positivo; esto solo resume variabilidad interna de la muestra observada.


### Limitaciones específicas

- El modelo fue ajustado con tuits de TASS 2020, no comentarios de YouTube; existe cambio de dominio.
- Macro-F1 0.705 implica errores sustantivos. No hay etiquetas humanas en estos datos para medir desempeño local, particularmente con español guatemalteco.
- Sarcasmo, ironía, citas, ambigüedad, errores ortográficos, lenguaje ofensivo y mezcla de idiomas pueden alterar la predicción. Sentimiento no equivale a postura política, emoción, toxicidad, acuerdo ni veracidad.
- Los textos mayores de 128 tokens se truncan. La tabla de control cuantifica cuántos fueron afectados.
- Las probabilidades del modelo no están calibradas con esta muestra; `confianza` no es garantía de corrección.
- Los grupos pequeños producen porcentajes inestables; por eso se exige n ≥ 5 y se añaden intervalos para n ≥ 10. Comentarios de un mismo autor o video tampoco son observaciones necesariamente independientes.
- Solo hay comentarios recolectados para 19 de 293 videos. La selección por consultas, cobertura parcial y concentración en pocos videos impiden generalizar a YouTube, Guatemala o las audiencias completas de los canales.

**Referencias:** Pérez et al. (2021), *pysentimiento: A Python Toolkit for Opinion Mining and Social NLP Tasks*, arXiv:2106.09462; Pérez et al. (2022), *RoBERTuito: a pre-trained language model for social media text in Spanish*, LREC 2022; García-Vega et al. (2020), *Overview of TASS 2020*.

## Conclusión

El análisis asigna sentimiento a todos los comentarios y conserva probabilidades para no ocultar ambigüedad. Las comparaciones permiten caracterizar videos, canales, categorías y comunidades con suficiente información, pero son descriptivas y asociativas. Los resultados deben leerse junto con los tamaños de grupo, intervalos, confianza del modelo y auditoría cualitativa; no permiten inferir causalidad ni opinión poblacional.